# Email Tone Rewriter

A Gradio-powered tool that rewrites emails in different tones, with real-time streaming output.

**What this project demonstrates:**
- Gradio UI with multiple input components (Textbox, Slider, Dropdown)
- Streaming responses for real-time text generation
- Dynamic system prompts based on user-selected options
- Practical use case for LLMs in everyday communication

Built as part of Module 3 (Gradio & Streaming) of the LLM & Agentic AI Bootcamp.

## Setup

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client ready!")

/Users/eugenionex/Downloads/LLM and Agentic AI Bootcamp Materials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Client ready!


## Tone Configuration

The slider maps to different tone descriptions that modify the system prompt dynamically.
This is the same technique we learned in the module with explanation levels.

In [2]:
TONE_LEVELS = {
    1: "very casual and friendly, like texting a close friend",
    2: "informal but polite, like writing to a colleague you know well",
    3: "neutral and professional, standard business tone",
    4: "formal and respectful, like writing to a senior manager or client",
    5: "very formal and diplomatic, like official corporate communication"
}

LANGUAGES = ["English", "Italian", "Spanish", "French", "German"]

print("Tones configured:")
for level, desc in TONE_LEVELS.items():
    print(f"  {level}: {desc}")

Tones configured:
  1: very casual and friendly, like texting a close friend
  2: informal but polite, like writing to a colleague you know well
  3: neutral and professional, standard business tone
  4: formal and respectful, like writing to a senior manager or client
  5: very formal and diplomatic, like official corporate communication


## Rewriter Function with Streaming

This is a **generator function** (uses `yield` instead of `return`).
Gradio detects this and updates the output in real-time as chunks arrive from the API.

Key concepts:
- `stream=True` in the API call enables chunk-by-chunk responses
- `yield` sends each partial result to the Gradio UI immediately
- The system prompt changes dynamically based on slider + dropdown selection

In [3]:
def rewrite_email(email_text, tone_level, language):
    """Rewrite an email in the selected tone with streaming output."""
    if not email_text.strip():
        yield "Please paste an email to rewrite."
        return
    
    tone_description = TONE_LEVELS.get(tone_level, TONE_LEVELS[3])
    
    system_prompt = f"""You are an expert email rewriter. Your task:
1. Rewrite the email the user provides in a tone that is {tone_description}
2. Write the rewritten email in {language}
3. Keep the same meaning and key information
4. Adjust greetings, closings, and phrasing to match the tone
5. Output ONLY the rewritten email, no explanations"""
    
    try:
        stream = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": email_text}
            ],
            temperature=0.7,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta and chunk.choices[0].delta.content:
                full_response += chunk.choices[0].delta.content
                yield full_response
    
    except Exception as e:
        yield f"Error: {e}"

## Gradio Interface

We combine multiple input components:
- **Textbox** for the email
- **Slider** for tone level (1-5)
- **Dropdown** for output language

The output uses `gr.Markdown` for nicely formatted streaming text.

In [4]:
demo = gr.Interface(
    fn=rewrite_email,
    inputs=[
        gr.Textbox(
            lines=8,
            placeholder="Paste your email here...",
            label="Original Email"
        ),
        gr.Slider(
            minimum=1,
            maximum=5,
            step=1,
            value=3,
            label="Tone (1=Casual, 3=Professional, 5=Very Formal)"
        ),
        gr.Dropdown(
            choices=LANGUAGES,
            value="English",
            label="Output Language"
        )
    ],
    outputs=gr.Markdown(
        label="Rewritten Email",
        container=True,
        height=300
    ),
    title="Email Tone Rewriter",
    description="Paste an email, choose the tone and language, and get it rewritten in real-time.",
    flagging_mode="never"
)

print("Launching Email Tone Rewriter...")
demo.launch()

Launching Email Tone Rewriter...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Try It!

Paste this sample email and try different tones:

```
Hi,

I wanted to follow up on the meeting we had last week about the project timeline.
I think we need to push the deadline back by two weeks because the dev team is
still working on the API integration. Can we schedule a call to discuss?

Thanks,
Alex
```

- **Tone 1 (Casual):** See how it becomes like a quick message to a friend
- **Tone 3 (Professional):** Standard business communication
- **Tone 5 (Very Formal):** Corporate/diplomatic language
- Try switching the language to Italian or Spanish!

## Key Takeaways

1. **Gradio components** (Textbox, Slider, Dropdown) make it easy to build interactive UIs
2. **Streaming** (`stream=True` + `yield`) gives real-time feedback - better UX than waiting
3. **Dynamic system prompts** let the same function behave differently based on user choices
4. **Generator functions** are how Gradio handles streaming - `yield` partial results instead of `return`
5. Multiple inputs are passed as a **list** to `gr.Interface`